## Bank Marketing Dataset — Full Training

In [ ]:
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [ ]:
data = pd.read_csv("bank-full.csv", sep=";")
print("Shape:", data.shape)
data.head(3)

## Task 1: Train Test Split

In [ ]:
X = data.drop("y", axis=1)
y = data["y"].map({"no": 0, "yes": 1})

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print("Train:", X_train.shape)
print("Test: ", X_test.shape)

## Task 2: One Hot Encoding

In [ ]:
X_train = pd.get_dummies(X_train)
X_test = pd.get_dummies(X_test)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

print("Columns after one hot:", X_train.shape[1])

## Task 3: NumPy Array to Tensor

In [ ]:
X_train_np = X_train.values.astype("float32")
X_test_np = X_test.values.astype("float32")
y_train_np = y_train.values.astype("float32").reshape(-1, 1)
y_test_np = y_test.values.astype("float32").reshape(-1, 1)

scaler = StandardScaler()
X_train_np = scaler.fit_transform(X_train_np).astype("float32")
X_test_np = scaler.transform(X_test_np).astype("float32")

print("NumPy array:", X_train_np.shape)

In [ ]:
X_train_tensor = torch.from_numpy(X_train_np)
X_test_tensor = torch.from_numpy(X_test_np)
y_train_tensor = torch.from_numpy(y_train_np)
y_test_tensor = torch.from_numpy(y_test_np)

print("X train tensor:", X_train_tensor.shape)
print("y train tensor:", y_train_tensor.shape)

### dataset, dataloader

In [ ]:
from torch.utils.data import TensorDataset

train_ds = TensorDataset(X_train_tensor, y_train_tensor)
test_ds = TensorDataset(X_test_tensor, y_test_tensor)
print("Dataset ready:", len(train_ds), "train rows")

In [ ]:
from torch.utils.data import DataLoader

batch_size = 32
train_dl = DataLoader(train_ds, batch_size, shuffle=True)

for xb, yb in train_dl:
    print("Batch X:", xb.shape)
    print("Batch y:", yb.shape)
    break

## nn.Linear

In [ ]:
import torch.nn as nn

model = nn.Linear(51, 1)
print(model.weight.shape)

## Loss function

In [ ]:
import torch.nn.functional as F

loss_fn = F.mse_loss

loss = loss_fn(model(X_train_tensor), y_train_tensor)
print("Start loss:", loss.item())

## OPTIMIZE

In [ ]:
opt = torch.optim.SGD(model.parameters(), lr=0.01)
opt

### now everything is set for model training

In [ ]:
def fit(num_epochs, model, loss_fn, opt, train_dl):

    for epoch in range(num_epochs):

        for xb, yb in train_dl:
            pred = model(xb)
            loss = loss_fn(pred, yb)
            loss.backward()
            opt.step()
            opt.zero_grad()

        if (epoch + 1) % 5 == 0:
            print('Epoch [{}/{}], Loss: {:.4f}'.format(epoch + 1, num_epochs, loss.item()))

In [ ]:
fit(20, model, loss_fn, opt, train_dl)

In [ ]:
preds = model(X_test_tensor)

print("Pehli 5 predictions:")
print(preds[:5].flatten())
print("Asli jawab (y test):")
print(y_test_tensor[:5].flatten())